# Feature Store Setup

Creates entities and managed feature views in `SB_COMMAND_CENTER.FEATURE_STORE`.


In [ ]:
from snowflake.snowpark import Session
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

session = Session.builder.configs({"connection_name": "my_connection"}).create()
session.use_database('SB_COMMAND_CENTER')
session.use_schema('FEATURE_STORE')

fs = FeatureStore(
    session=session,
    database='SB_COMMAND_CENTER',
    name='FEATURE_STORE',
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
print("FeatureStore initialized")

## Create Entities


In [ ]:
customer_entity = Entity(
    name='CUSTOMER',
    join_keys=['CUSTOMER_ID'],
    desc='Customer identifier for omnichannel transactions'
)
fs.register_entity(customer_entity)

retailer_product_week_entity = Entity(
    name='RETAILER_PRODUCT_WEEK',
    join_keys=['RETAILER_NAME', 'ITEM_BRAND', 'RETAILER_DATE'],
    desc='Retailer x Brand x Week grain for POS sell-through'
)
fs.register_entity(retailer_product_week_entity)

entities = fs.list_entities().collect()
for e in entities:
    print(f"  Entity: {e['NAME']}  keys={e['JOIN_KEYS']}")

## Feature View: CUSTOMER_RFM_FV

RFM features from `RAW.CUSTOMER_TRANSACTIONS`.


In [ ]:
customer_rfm_df = session.sql("""
SELECT
    CUSTOMER_ID,
    DATEDIFF('day', MAX(TRANSACTION_DATE), CURRENT_TIMESTAMP()) AS RECENCY_DAYS,
    COUNT(*) AS ORDER_FREQUENCY,
    SUM(PURCHASE_AMOUNT) AS TOTAL_MONETARY,
    AVG(PURCHASE_AMOUNT) AS AVG_ORDER_VALUE,
    COUNT(DISTINCT BRAND) AS BRAND_DIVERSITY,
    SUM(CASE WHEN DISCOUNT_OFFER_APPLIED IS NOT NULL THEN 1 ELSE 0 END)::FLOAT
        / NULLIF(COUNT(*), 0) AS DISCOUNT_SENSITIVITY,
    MAX(CASE WHEN PROREWARDS_CUSTOMER = 'Y' THEN 1 ELSE 0 END) AS PROREWARDS_FLAG
FROM SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS
GROUP BY CUSTOMER_ID
""")

customer_rfm_fv = FeatureView(
    name='CUSTOMER_RFM_FV',
    entities=[customer_entity],
    feature_df=customer_rfm_df,
    refresh_freq='0 6 * * * America/Chicago',
    desc='RFM features: recency, frequency, monetary, brand diversity, discount sensitivity'
)

customer_rfm_fv = fs.register_feature_view(
    feature_view=customer_rfm_fv,
    version='v1',
    block=True
)
print("Registered CUSTOMER_RFM_FV v1")

## Feature View: CUSTOMER_BEHAVIOR_FV

Behavioral features from web orders + lines.


In [ ]:
customer_behavior_df = session.sql("""
WITH order_agg AS (
    SELECT
        o.CUSTOMER_ID,
        o.ORDER_ID,
        o.BASE_GRAND_TOTAL,
        o.COUPON_CODE,
        o.SHIPPING_STATE,
        o.SHIPPING_COUNTRY,
        COUNT(l.ITEM_ID) AS items_in_order
    FROM SB_COMMAND_CENTER.RAW.WEB_ORDERS o
    JOIN SB_COMMAND_CENTER.RAW.WEB_ORDER_LINES l ON o.ORDER_ID = l.ORDER_ID
    WHERE o.STATUS != 'canceled'
    GROUP BY o.CUSTOMER_ID, o.ORDER_ID, o.BASE_GRAND_TOTAL, o.COUPON_CODE,
             o.SHIPPING_STATE, o.SHIPPING_COUNTRY
)
SELECT
    CUSTOMER_ID,
    AVG(BASE_GRAND_TOTAL) AS AVG_BASKET_SIZE,
    AVG(items_in_order) AS AVG_BASKET_ITEMS,
    SUM(CASE WHEN COUPON_CODE IS NOT NULL THEN 1 ELSE 0 END)::FLOAT
        / NULLIF(COUNT(*), 0) AS COUPON_USAGE_RATE,
    MODE(SHIPPING_STATE) AS GEOGRAPHY_STATE,
    MODE(SHIPPING_COUNTRY) AS GEOGRAPHY_COUNTRY
FROM order_agg
GROUP BY CUSTOMER_ID
""")

customer_behavior_fv = FeatureView(
    name='CUSTOMER_BEHAVIOR_FV',
    entities=[customer_entity],
    feature_df=customer_behavior_df,
    refresh_freq='0 6 * * * America/Chicago',
    desc='Web behavior: basket size, coupon usage, geography'
)

customer_behavior_fv = fs.register_feature_view(
    feature_view=customer_behavior_fv,
    version='v1',
    block=True
)
print("Registered CUSTOMER_BEHAVIOR_FV v1")

## Feature View: POS_WEEKLY_SALES_FV

Weekly POS metrics at Retailer x Brand x Week grain.


In [ ]:
pos_weekly_df = session.sql("""
SELECT
    RETAILER_NAME,
    ITEM_BRAND,
    RETAILER_DATE,
    SUM(GROSS_AMT_SOLD_UNITS) AS GROSS_UNITS_SOLD,
    SUM(NET_SALES_RETAIL) AS NET_SALES_RETAIL,
    SUM(NET_SALES_UNITS) AS NET_SALES_UNITS,
    SUM(INVENTORY_UNITS) AS INVENTORY_ON_HAND,
    SUM(ON_ORDER_QUANTITY) AS ON_ORDER_QUANTITY,
    CASE WHEN SUM(GROSS_SALES_RETAIL) > 0
        THEN SUM(TOTAL_MARKDOWN) / SUM(GROSS_SALES_RETAIL) ELSE 0
    END AS MARKDOWN_PCT,
    CASE WHEN SUM(GROSS_AMT_SOLD_UNITS) > 0
        THEN SUM(CUSTOMER_RETURN_UNITS) / SUM(GROSS_AMT_SOLD_UNITS) ELSE 0
    END AS RETURN_RATE,
    CASE WHEN SUM(INVENTORY_UNITS) > 0
        THEN SUM(GROSS_AMT_SOLD_UNITS) / (SUM(GROSS_AMT_SOLD_UNITS) + SUM(INVENTORY_UNITS)) ELSE 0
    END AS SELL_THROUGH_RATE
FROM SB_COMMAND_CENTER.RAW.POS_SALES
GROUP BY RETAILER_NAME, ITEM_BRAND, RETAILER_DATE
""")

pos_weekly_fv = FeatureView(
    name='POS_WEEKLY_SALES_FV',
    entities=[retailer_product_week_entity],
    feature_df=pos_weekly_df,
    refresh_freq='0 4 * * 1 America/Chicago',
    desc='Weekly POS: units, revenue, inventory, markdown, return rate, sell-through'
)

pos_weekly_fv = fs.register_feature_view(
    feature_view=pos_weekly_fv,
    version='v1',
    block=True
)
print("Registered POS_WEEKLY_SALES_FV v1")

## Verify


In [ ]:
fvs = fs.list_feature_views().collect()
print(f"Feature Views: {len(fvs)}")
for fv in fvs:
    print(f'  {fv["NAME"]}${fv["VERSION"]}: {fv["DESC"]}')

entities = fs.list_entities().collect()
print(f"\nEntities: {len(entities)}")
for e in entities:
    print(f'  {e["NAME"]}: keys={e["JOIN_KEYS"]}')